In [15]:
from dotenv import load_dotenv
import os

In [16]:
import requests

In [17]:
load_dotenv()
socks_proxy = os.getenv('PROXY')


In [18]:
session = requests.Session()
user_agent = 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_14_3) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/72.0.3626.121 Safari/537.36'
headers = {'user-agent' : user_agent}
proxies = {
    'http' : f'socks5h://{socks_proxy}',
    'https' : f'socks5h://{socks_proxy}',
}

locationIdResponse = session.get('https://sigmas.social.gouv.fr/server/rest/services/baignades/fra_vue_baignade/MapServer/5/query?f=json&where=1%3D1&returnIdsOnly=true&geometry=-9678710.360800,-28545813.431743,30099965.362980,42759085.476384&geometryType=esriGeometryEnvelope&spatialRel=esriSpatialRelEnvelopeIntersects', headers=headers, proxies=proxies).json()

In [19]:
idList = locationIdResponse['objectIds']

In [20]:
from itertools import islice

In [21]:
it = iter(idList) 
n = 100
res = [list(islice(it, n)) for _ in range((len(idList) + n - 1) // n)]

In [22]:
dataset = []
for item in res:
    item = [str(x) for x in item]
    ids = ",".join(item)
    data = session.get(f"https://sigmas.social.gouv.fr/server/rest/services/baignades/fra_vue_baignade/MapServer/5/query?f=geojson&objectIds={ids}&inSR&outSR&returnGeometry=true&outFields=*&returnM=false&returnZ=false", headers=headers, proxies=proxies).json()
    dataset = dataset  + data['features']

In [23]:
import pandas as pd

In [28]:
df = pd.json_normalize(dataset)

In [29]:
df = df[[
    "properties.OBJECTID",
    "properties.lsite",
    "geometry.coordinates"
]]

In [30]:
df.rename(mapper={
    "properties.OBJECTID": "id",
    "properties.lsite": "name",
    "geometry.coordinates": "coordinates"
}, axis=1, inplace=True)
df['lon'] = [item[0] for item in df['coordinates']]
df['lat'] = [item[1] for item in df['coordinates']]
df['name'] = df['name'].str.title()
df

,id,name,coordinates,lon,lat
0,1,La Govelle,"[-2.4556977268936504, 47.26728240513849]",-2.455698,47.267282
1,2,Plaine Sud Du Parc De Choisy,"[2.4309229549256335, 48.760449220438844]",2.430923,48.760449
2,3,Montaubry Les Patins,"[4.5256678719055285, 46.78933051172293]",4.525668,46.789331
3,4,Ramberchamp-Kattendycke,"[6.852752881493553, 48.06674163518667]",6.852753,48.066742
4,5,Croix,"[-4.411277068379034, 48.638186410759275]",-4.411277,48.638186
...,...,...,...,...,...
15510,15511,La Naute,"[2.2812829935738845, 46.03349971141232]",2.281283,46.033500
15511,15512,Plan D'Eau Du Moulin De Savin,"[3.984045972898026, 44.93584254255614]",3.984046,44.935843
15512,15513,Plage Camping Bellevue,"[5.793558481126024, 45.55463862625989]",5.793558,45.554639
15513,15514,Plage Saint Alban,"[5.796720816827432, 45.56190341311844]",5.796721,45.561903
